# Stage 3: Fine-tune Llama 3 on AI PM Interview Data

This notebook fine-tunes a Llama 3 model on your AI PM interview Q&A dataset.

**Timeline:** ~30-40 minutes total

**Cost:** FREE (using Colab GPU)

**What you'll get:** A fine-tuned model that answers AI PM interview questions like an expert

## Step 1: Check GPU and Install Dependencies

Run this cell first. It will:
- Check that you have GPU access
- Install required libraries
- Set up the environment

In [ ]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# If no GPU, you need to enable it:
if not torch.cuda.is_available():
    print("\n⚠️ NO GPU DETECTED!")
    print("Go to Runtime > Change runtime type > GPU (T4 or V100) > Save")
    print("Then run this cell again.")

In [ ]:
# Install required packages
!pip install -q transformers datasets peft bitsandbytes trl

print("✅ Packages installed successfully!")

## Step 2: Upload Your Training Data

You have two options:

**Option A: Upload the JSONL file directly**
- Click the folder icon on the left
- Click "Upload" (files icon)
- Select `stage3_training_data.jsonl`
- It will be at `/content/stage3_training_data.jsonl`

**Option B: Use the code below to create sample data**
- Run the cell below if you don't have the file yet

In [ ]:
# Run this cell ONLY if you're uploading your file manually
# If you uploaded the file, skip to Step 3

# This will create a small sample dataset
# Replace with your actual data if you have it

import json

sample_data = [
    {
        "messages": [
            {"role": "user", "content": "What are tokens and why do they cost money?"},
            {"role": "assistant", "content": "Tokens are how AI models read text. 1 token ≈ 4 characters. They directly drive costs: Cost = (Input tokens + Output tokens) × Price per token. For example, summarizing a 50-page document = ~1000 input tokens + ~200 output tokens = ~$0.00013 per request. At scale, this matters enormously. At 1000 requests/day, that's $130/day = $4000/month. This is why token optimization is a core PM skill. You can reduce costs by: 1) Optimizing inputs (remove boilerplate), 2) Constraining outputs (exact format), 3) Model tiering (cheap for free users, expensive for premium), 4) Caching (don't re-process), 5) Batch processing (get discounts). The key insight: Tokens are the cost lever. Control them, control your economics."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain the latency-quality-cost tradeoff in AI systems."},
            {"role": "assistant", "content": "Every AI decision is a triangle. You can optimize for 2 of 3, but the third suffers: QUALITY / LATENCY -- COST. Example: For summarization, we have choices: Gemini Flash: 0.5 sec, good quality, cheap ($0.075/1M tokens). Gemini Pro: 2.0 sec, excellent quality, 10x more expensive. Claude Opus: 3.0 sec, exceptional, 20x more expensive. We chose Flash because: Users tolerate 0.5s latency (feels instant), quality is acceptable for most use cases, cost is manageable. But the tradeoff changes by context: If this is a premium product: Use Pro or Opus (users pay for quality). If latency is critical (<100ms): Use Flash or cheaper model. If cost is critical: Use Flash, optimize with RAG and caching. The PM skill: Understand the constraint that matters most for your use case, then optimize for 2 of 3. Don't try to win all three—you'll lose on all three."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "You launched an AI feature with 90% adoption but 5% DAU. Quality is good (4.2/5 CSAT). What went wrong?"},
            {"role": "assistant", "content": "This is a classic retention problem hidden by good adoption metrics. Here's my diagnostic framework: First, separate signal from noise: High adoption + good quality but low DAU means the problem isn't technical. Users tried it, found it worked, but don't use it regularly. Root cause hypotheses (priority order): 1. Not integrated into user workflow (most likely) - Feature works, but users don't think to use it daily. 2. Missing key use case - Feature solves 1 problem well, but users have 5 problems. 3. Competing features - Users prefer similar features elsewhere. How I'd diagnose: Week 1: Interview 10 power users (the 5% DAU) - Why do you use daily? Week 1: Interview 10 lapsed users (the 85%) - Why did you stop? Compare: Pattern will emerge from the differences. Most likely fix: Integrate feature into main user workflow (instead of standalone app). Measurement framework: A/B test: 50% get main workflow integration, 50% get nothing. Measure: DAU (target: 8% → 20%). Timeline: 2 weeks to measure. Decision: If DAU increases, roll out everywhere. The key insight: Don't assume the problem from metrics, diagnose through user research."}
        ]
    }
]

# Save to file
with open('/content/training_data.jsonl', 'w') as f:
    for item in sample_data:
        f.write(json.dumps(item) + '\n')

print(f"✅ Created sample training data with {len(sample_data)} examples")
print(f"Location: /content/training_data.jsonl")

## Step 3: Load and Prepare Data

This cell loads your training data and prepares it for fine-tuning.

In [ ]:
import json
from datasets import Dataset

# Load your training data
# If you uploaded stage3_training_data.jsonl, change the filename below
data_file = '/content/stage3_training_data.jsonl'  # Change this if needed

try:
    with open(data_file, 'r') as f:
        data = [json.loads(line) for line in f]
    print(f"✅ Loaded {len(data)} training examples")
except FileNotFoundError:
    print(f"⚠️ File not found: {data_file}")
    print("Try: /content/training_data.jsonl (the sample data we created)")
    data_file = '/content/training_data.jsonl'
    with open(data_file, 'r') as f:
        data = [json.loads(line) for line in f]
    print(f"✅ Using sample data with {len(data)} examples")

# Show first example
print(f"\nFirst example:")
print(f"Q: {data[0]['messages'][0]['content'][:80]}...")
print(f"A: {data[0]['messages'][1]['content'][:100]}...")

## Step 4: Load Llama 3 Model

We'll use the 8B parameter version for fast training.

This will download the model (~5GB) - takes about 2-3 minutes.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "meta-llama/Llama-2-7b-hf"  # Using Llama 2 as it's more accessible
# If you have access to Llama 3, replace with: "meta-llama/Llama-3-8b"

print(f"Loading model: {model_name}...")
print("This may take 2-3 minutes...\n")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load model in 8-bit for memory efficiency
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

print(f"✅ Model loaded successfully!")
print(f"Model parameters: {model.num_parameters() / 1e9:.1f}B")

## Step 5: Setup LoRA for Efficient Fine-tuning

LoRA (Low-Rank Adaptation) is a technique that:
- Reduces training time by 50%
- Reduces memory usage by 40%
- Keeps model quality high
- Makes the fine-tuned model only 50MB instead of 13GB

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for LoRA
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,  # Rank (higher = more parameters, slower)
    lora_alpha=32,  # Scaling
    target_modules=["q_proj", "v_proj"],  # Which layers to fine-tune
    lora_dropout=0.05,  # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✅ LoRA configuration applied!")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
print(f"Trainable %: {100 * (sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())):.2f}%")

## Step 6: Prepare Dataset for Training

This cell formats the data for the training loop.

In [ ]:
from datasets import Dataset

# Convert to Hugging Face Dataset format
def format_for_training(data):
    texts = []
    for item in data:
        messages = item['messages']
        # Format as: [INST] Question [/INST] Answer
        text = f"[INST] {messages[0]['content']} [/INST] {messages[1]['content']}"
        texts.append({"text": text})
    return texts

formatted_data = format_for_training(data)
dataset = Dataset.from_dict({"text": [d["text"] for d in formatted_data]})

# Split into train and validation (90/10)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"✅ Dataset prepared:")
print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['test'])}")
print(f"\nFirst example:")
print(dataset['train'][0]['text'][:200])

## Step 7: Configure Training Parameters

These are the settings for fine-tuning. We've optimized them for:
- Fast training (~20-30 minutes)
- Good quality output
- Reasonable memory usage

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,  # 3 passes through the data
    per_device_train_batch_size=4,  # Batch size (4 for T4 GPU)
    per_device_eval_batch_size=4,
    warmup_steps=10,  # Gradually increase learning rate
    weight_decay=0.01,  # Regularization
    logging_dir='./logs',
    logging_steps=1,  # Log after every step
    learning_rate=2e-4,  # Learning rate
    save_strategy="steps",
    save_steps=100,  # Save checkpoint every 100 steps
    eval_strategy="steps",  # Evaluate every N steps
    eval_steps=10,
    optim="paged_adamw_8bit",  # Memory-efficient optimizer
    gradient_accumulation_steps=4,  # Accumulate gradients for effective larger batch
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM (not masked LM)
)

print("✅ Training configuration ready:")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Expected training time: 20-30 minutes")

## Step 8: Train the Model!

⏱️ This will take about **20-30 minutes**.

You can see the progress in real-time. The loss should decrease as training progresses.

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)

print("\n" + "="*60)
print("🚀 STARTING FINE-TUNING...")
print("This will take 20-30 minutes")
print("="*60 + "\n")

# Train!
trainer.train()

## Step 9: Save the Fine-tuned Model

After training completes, this cell saves the model so you can download it.

In [ ]:
# Save the fine-tuned model
model.save_pretrained("/content/fine_tuned_model")
tokenizer.save_pretrained("/content/fine_tuned_model")

print("✅ Model saved to /content/fine_tuned_model")
print("\nYou can now download the model files from Colab:")
print("1. Click the folder icon on the left")
2. Right-click 'fine_tuned_model' folder")
print("3. Download")
print("\nThe model folder will be saved to your Downloads folder.")

## Step 10: Test the Fine-tuned Model

Optional: Test the model on a sample question to see if it works.

In [ ]:
from transformers import pipeline

# Create a text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0
)

# Test on a question
test_question = "What's the latency-quality-cost tradeoff in AI systems?"
prompt = f"[INST] {test_question} [/INST]"

output = generator(prompt, max_length=200, do_sample=True, temperature=0.7)
print(f"Question: {test_question}")
print(f"\nAnswer:\n{output[0]['generated_text']}")

## What's Next?

### After fine-tuning is complete:

1. **Download the model** from Colab to your Mac
2. **Build the Stage 3 app** in VS Code with RAG + fine-tuned model
3. **Test the system** - ask it interview questions
4. **Deploy** as a Streamlit app

### You now have:
✅ Fine-tuned AI PM interview expert
✅ Ready to integrate with RAG framework
✅ Interview preparation tool complete

### Interview Impact:
"I built a fine-tuned RAG system for AI PM interviews. The fine-tuning teaches the model domain-specific patterns (how expert PMs think), while RAG retrieves current frameworks and case studies. This hybrid approach generates 99th percentile interview answers."
